# Content Moderation SaaS — DistilBERT Fine-Tuning

Fine-tunes DistilBERT for multi-label toxic comment classification on the Jigsaw Toxic Comment Classification dataset.

**Before running:** Runtime → Change runtime type → select **GPU** (training is faster here even though deployment runs on CPU).

**Labels:** toxic, severe_toxic, obscene, threat, insult, identity_hate (multi-label — a comment can have more than one).

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn kaggle accelerate

## 2. Upload Kaggle API credentials

Get `kaggle.json` from https://www.kaggle.com/settings → API → Create New Token, then upload it below.

In [ ]:
import os

# Store this in Colab Secrets as KAGGLE_API_TOKEN instead of separate username/key
from google.colab import userdata
kaggle_token = userdata.get('KAGGLE_API_TOKEN')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(kaggle_token)

import subprocess
subprocess.run(['chmod', '600', '/root/.kaggle/access_token'])

CompletedProcess(args=['chmod', '600', '/root/.kaggle/access_token'], returncode=0)

In [ ]:
!kaggle competitions list

ref                                                                           deadline             category         reward  teamCount  userHasEntered  
----------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge   2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                            2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                    2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection          2023-06-14 23:59:00  Featured  1,000,000 Usd       1249           False  
https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3                  2026-11-02

## 3. Download the Jigsaw dataset

Note: you must accept the competition rules on the Kaggle website first (visit the competition page and click 'Join Competition'), or this download will fail with a 403 error.

In [ ]:
!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge
!unzip -q jigsaw-toxic-comment-classification-challenge.zip -d jigsaw_data
!unzip -q jigsaw_data/train.csv.zip -d jigsaw_data

100% 52.6M/52.6M [00:03<00:00, 15.9MB/s]



## 4. Load and sample the data

Using a stratified sample rather than the full ~160k rows to keep training time reasonable given the coursework deadline. Adjust `SAMPLE_SIZE` down further if training is still too slow.

In [ ]:
import pandas as pd

LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
SAMPLE_SIZE = 100000

df = pd.read_csv("jigsaw_data/train.csv")
print("Full dataset size:", len(df))
print(df[LABEL_COLS].sum())

# Keep every row with at least one positive label (rare classes), then top up with clean comments
positive_rows = df[df[LABEL_COLS].sum(axis=1) > 0]
negative_rows = df[df[LABEL_COLS].sum(axis=1) == 0]

n_negative_needed = max(SAMPLE_SIZE - len(positive_rows), 0)
sampled_negative = negative_rows.sample(n=min(n_negative_needed, len(negative_rows)), random_state=42)

df_sample = pd.concat([positive_rows, sampled_negative]).sample(frac=1, random_state=42).reset_index(drop=True)
print("Sampled dataset size:", len(df_sample))
print(df_sample[LABEL_COLS].sum())

Full dataset size: 159571
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64
Sampled dataset size: 100000
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64


## 5. Train / validation / test split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df_sample, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

Train: 80000 Val: 10000 Test: 10000


## 6. Tokenization

In [ ]:
from transformers import DistilBertTokenizerFast
import torch
from torch.utils.data import Dataset

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class ToxicCommentsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

train_dataset = ToxicCommentsDataset(
    train_df["comment_text"].tolist(), train_df[LABEL_COLS].values, tokenizer
)
val_dataset = ToxicCommentsDataset(
    val_df["comment_text"].tolist(), val_df[LABEL_COLS].values, tokenizer
)
test_dataset = ToxicCommentsDataset(
    test_df["comment_text"].tolist(), test_df[LABEL_COLS].values, tokenizer
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## 7. Model setup — freeze most layers

Only the classification head and the last transformer layer are trainable. This is faster to train and gives you a genuine parametrisation story for the report (you can discuss why you froze most of the backbone and only fine-tuned the top layer).

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_COLS),
    problem_type="multi_label_classification",
)

# Freeze all DistilBERT transformer layers except the last one
for name, param in model.distilbert.named_parameters():
    if "layer.5" not in name:  # DistilBERT has 6 layers (0-5); keep only the last trainable
        param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.1f}%)")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 7,683,078 / 66,958,086 (11.5%)


## 8. Training

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # sigmoid, since this is multi-label
    preds = (probs > 0.5).astype(int)

    try:
        auc = roc_auc_score(labels, probs, average="macro")
    except ValueError:
        auc = float("nan")  # can happen if a label has no positive examples in a batch

    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return {"roc_auc": auc, "f1_macro": f1}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Roc Auc,F1 Macro
1,0.059969,0.065659,0.977766,0.486291
2,0.067247,0.063401,0.982178,0.596905
3,0.059677,0.060407,0.983154,0.593258


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15000, training_loss=0.06977800064086914, metrics={'train_runtime': 506.8935, 'train_samples_per_second': 473.472, 'train_steps_per_second': 29.592, 'total_flos': 7948610887680000.0, 'train_loss': 0.06977800064086914, 'epoch': 3.0})

## 9. Evaluate on the held-out test set

In [ ]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

Training Loss,Validation Loss,Epoch,Roc Auc,F1 Macro
0.059677,0.057589,3,0.984510,0.622128


{'eval_loss': 0.057589348405599594, 'eval_roc_auc': 0.9845097221474347, 'eval_f1_macro': 0.6221275612509346}


## 10. Save the fine-tuned model

This produces the files your `inference-service` will load. Zip and download, or push to the HuggingFace Hub (recommended — makes it easy for `inference-service`'s Docker build step to fetch automatically).

In [ ]:
SAVE_PATH = "./fine-tuned-toxicity-distilbert"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved to", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./fine-tuned-toxicity-distilbert


### Option A: Download as a zip

In [ ]:
from google.colab import files

!zip -r fine-tuned-toxicity-distilbert.zip fine-tuned-toxicity-distilbert
files.download("fine-tuned-toxicity-distilbert.zip")

  adding: fine-tuned-toxicity-distilbert/ (stored 0%)
  adding: fine-tuned-toxicity-distilbert/model.safetensors (deflated 8%)
  adding: fine-tuned-toxicity-distilbert/config.json (deflated 54%)
  adding: fine-tuned-toxicity-distilbert/tokenizer_config.json (deflated 43%)
  adding: fine-tuned-toxicity-distilbert/tokenizer.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Option B (recommended): Push to HuggingFace Hub

Makes it trivial for `inference-service`'s Dockerfile to download the model automatically on build (`from_pretrained("your-username/model-name")`), satisfying the brief's "missing dependencies downloaded during build/up" requirement.

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login

login()  # paste your HuggingFace token when prompted (from https://huggingface.co/settings/tokens)

REPO_NAME = "moboluw4rin/toxicity-distilbert"  # replace with your HuggingFace username

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print(f"Model pushed to https://huggingface.co/{REPO_NAME}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mmil_s5/model.safetensors:   3%|2         | 6.88MB /  268MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Model pushed to https://huggingface.co/moboluw4rin/toxicity-distilbert


## 11. Quick sanity check — run a few example predictions

In [ ]:
import torch

test_comments = [
    "Thanks for the helpful explanation, really appreciated!",
    "You are an idiot and should shut up forever.",
    "I disagree with this edit but let's discuss it calmly.",
]

model.eval()
for comment in test_comments:
    inputs = tokenizer(comment, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LENGTH)
    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.sigmoid(logits).squeeze().tolist()
    scores = dict(zip(LABEL_COLS, [round(p, 3) for p in probs]))
    print(f"\n'{comment}'\n  -> {scores}")


'Thanks for the helpful explanation, really appreciated!'
  -> {'toxic': 0.0, 'severe_toxic': 0.0, 'obscene': 0.0, 'threat': 0.0, 'insult': 0.0, 'identity_hate': 0.0}

'You are an idiot and should shut up forever.'
  -> {'toxic': 0.993, 'severe_toxic': 0.092, 'obscene': 0.728, 'threat': 0.034, 'insult': 0.949, 'identity_hate': 0.015}

'I disagree with this edit but let's discuss it calmly.'
  -> {'toxic': 0.0, 'severe_toxic': 0.0, 'obscene': 0.0, 'threat': 0.0, 'insult': 0.0, 'identity_hate': 0.0}


## Notes for the report

- **Dataset**: Jigsaw Toxic Comment Classification (Kaggle), stratified sample of ~30k rows from the full ~160k to fit the coursework timeline.
- **Model**: DistilBERT (`distilbert-base-uncased`), fine-tuned for multi-label classification across 6 toxicity categories.
- **Parametrisation choice**: only the final transformer layer + classification head were unfrozen, keeping most of DistilBERT's pretrained language understanding intact while adapting to the toxicity task — faster to train and reduces overfitting risk on a relatively small sample.
- **Evaluation metric**: macro-averaged ROC-AUC (appropriate for multi-label, imbalanced classes) and macro F1.
- **Threshold**: predictions use a 0.5 sigmoid threshold per label — mention this explicitly, and consider discussing what changing this threshold would do to precision/recall trade-offs in your Limitations section.